# 多时期外汇套利问题 - 不同初始货币分析

本notebook演示如何使用COPT求解器来解决多时期外汇套利问题。我们将分析从不同初始货币开始的套利机会。

## 问题描述

- **货币种类**: 美元、欧元、英镑、日元
- **时期数**: T = 4
- **目标**: 从1单位初始货币开始，通过多次兑换，最大化最终回到初始货币的总量

## 数学模型

**决策变量**:
- $x_{i,j,t}$: 在时期t从货币i兑换到货币j的数量
- $y_{i,t}$: 在时期t持有货币i的数量

**目标函数**:
$$\max y_{s,T}$$

**约束条件**:
1. 初始条件: $y_{s,0} = 1$, $y_{i,0} = 0$ (对于 $i \neq s$)
2. 流量约束: $\sum_{j \neq i} x_{i,j,t} \leq y_{i,t}$
3. 动态方程: $y_{i,t+1} = y_{i,t} - \sum_{j \neq i} x_{i,j,t} + \sum_{j \neq i} r_{j,i} \cdot x_{j,i,t}$

## 1. 导入必要的库

In [7]:
import coptpy as cp
from coptpy import COPT
import numpy as np

## 2. 定义问题参数

首先定义货币种类、时期数和汇率矩阵。

In [8]:
# 货币编号：1=美元，2=欧元，3=英镑，4=日元
currencies = [1, 2, 3, 4]
currency_names = {1: "美元", 2: "欧元", 3: "英镑", 4: "日元"}

# 时期数
T = 4

# 汇率矩阵 r[i][j] 表示从货币i兑换到货币j的汇率
r = {
    (1, 2): 1.1486, (1, 3): 0.7003, (1, 4): 133.38,
    (2, 1): 0.8706, (2, 3): 0.6097, (2, 4): 116.12,
    (3, 1): 1.4279, (3, 2): 1.6401, (3, 4): 190.45,
    (4, 1): 0.00750, (4, 2): 0.00861, (4, 3): 0.00525
}

print("="*70)
print("多时期外汇套利问题 - 不同初始货币分析")
print("="*70)
print(f"时期数: T = {T}")
print(f"货币种类: {', '.join([currency_names[c] for c in currencies])}")

多时期外汇套利问题 - 不同初始货币分析
时期数: T = 4
货币种类: 美元, 欧元, 英镑, 日元


## 3. 定义求解函数

创建一个函数来为指定的初始货币构建和求解优化模型。

In [13]:
def solve_arbitrage_for_currency(s):
    """
    为指定的初始货币s求解套利问题
    
    参数:
        s: 初始货币编号 (1=美元, 2=欧元, 3=英镑, 4=日元)
    
    返回:
        result: 包含求解结果的字典
    """
    print(f"\n{'='*70}")
    print(f"初始货币: {currency_names[s]}")
    print(f"{'='*70}")
    
    # 步骤1: 创建COPT环境和模型
    env = cp.Envr()
    model = env.createModel(f"multiperiod_arbitrage_start_{s}")
    
    # 步骤2: 定义决策变量
    # x[i, j, t]: 在时期t从货币i兑换到货币j的数量
    x = {}
    for i in currencies:
        for j in currencies:
            if i != j:
                for t in range(T):
                    x[i, j, t] = model.addVar(lb=0, name=f"x_{i}_{j}_{t}")
    
    # y[i, t]: 在时期t持有货币i的数量
    y = {}
    for i in currencies:
        for t in range(T + 1):
            y[i, t] = model.addVar(lb=0, name=f"y_{i}_{t}")
    
    # 步骤3: 设置初始条件
    # 初始时刻，只有起始货币s有1单位，其他货币为0
    model.addConstr(y[s, 0] == 1, name="initial_s")
    for i in currencies:
        if i != s:
            model.addConstr(y[i, 0] == 0, name=f"initial_{i}")
    
    # 步骤4: 设置目标函数
    # 最大化最终回到起始货币的总量
    obj_expr = y[s, T]
    model.setObjective(obj_expr, COPT.MAXIMIZE)
    
    # 步骤5: 添加约束条件
    # 约束1: 流量约束 - 从货币i兑换出去的总量不能超过持有量
    for i in currencies:
        for t in range(T):
            outflow = cp.quicksum(x[i, j, t] for j in currencies if j != i)
            model.addConstr(outflow <= y[i, t], name=f"flow_limit_{i}_{t}")
    
    # 约束2: 动态方程 - 描述货币持有量的演化
    for i in currencies:
        for t in range(T):
            # 流出: 从货币i兑换到其他货币
            outflow = cp.quicksum(x[i, j, t] for j in currencies if j != i)
            # 流入: 从其他货币兑换到货币i (乘以汇率)
            inflow = cp.quicksum(x[j, i, t] * r[j, i] for j in currencies if j != i)
            # 下一时期的持有量 = 当前持有量 - 流出 + 流入
            model.addConstr(y[i, t+1] == y[i, t] - outflow + inflow, 
                           name=f"dynamics_{i}_{t}")
    
    # 步骤6: 求解模型
    model.solve()
    
    # 步骤7: 处理求解结果
    result = {}
    
    if model.status == COPT.OPTIMAL:
        obj_val = model.objVal
        profit = obj_val - 1
        profit_rate = profit * 100
        
        result = {
            'status': 'OPTIMAL',
            'obj_val': obj_val,
            'profit': profit,
            'profit_rate': profit_rate,
            'x': x,
            'y': y
        }
        
        print(f"求解状态: 最优")
        print(f"最优目标函数值: {obj_val:.10f}")
        print(f"套利收益: {profit:.10f}")
        print(f"套利收益率: {profit_rate:.6f}%")
        
        # 输出非零的兑换活动
        print(f"\n兑换活动（非零项）:")
        for t in range(T):
            has_activity = False
            for i in currencies:
                for j in currencies:
                    if i != j and (i, j, t) in x:
                        x_val = x[i, j, t].X
                        if x_val > 1e-6:
                            if not has_activity:
                                print(f"  时期 t={t}:")
                                has_activity = True
                            print(f"    {currency_names[i]} → {currency_names[j]}: {x_val:.6f}")
    
    elif model.status == COPT.UNBOUNDED:
        result = {
            'status': 'UNBOUNDED',
            'obj_val': float('inf'),
            'profit': float('inf'),
            'profit_rate': float('inf')
        }
        print(f"求解状态: 无界 - 存在无限套利机会")
    
    else:
        result = {
            'status': str(model.status),
            'obj_val': None,
            'profit': None,
            'profit_rate': None
        }
        print(f"求解状态: {model.status}")
    
    return result

## 4. 求解所有初始货币的情况

对每种货币作为初始货币进行求解，比较不同起点的套利机会。

In [10]:
# 存储所有结果
results = {}

# 为每个货币作为起始货币进行求解
for s in currencies:
    results[s] = solve_arbitrage_for_currency(s)


初始货币: 美元
Cardinal Optimizer v8.0.3. Build date Jan 13 2026
Copyright Cardinal Operations 2026. All Rights Reserved

No license found. LP size is limited to 10000 variables and 10000 constraints
Please apply for a license from www.shanshu.ai/copt

Model fingerprint: e2c2d2dd

Using Cardinal Optimizer v8.0.3 on Windows (25H2 Build 26200 - x86_64)
The CPU model is Intel(R) Core(TM) i7-14650HX
Hardware has 16 physical cores and 24 logical cores. Using instruction set X86_AVX2 (10)
Maximizing an LP problem

The original problem has:
    36 rows, 68 columns and 196 non-zero elements
The presolved problem has:
    22 rows, 45 columns and 142 non-zero elements

Starting the simplex solver using up to 8 threads

Problem info:
    Range of matrix coefficients:    [6e-01,2e+00]
    Range of rhs coefficients:       [4e+00,4e+00]
    Range of bound coefficients:     [0e+00,0e+00]
    Range of cost coefficients:      [3e-01,3e-01]

Method   Iteration           Objective  Primal.NInf   Dual.NInf    

## 5. 汇总结果

以表格形式展示所有初始货币的求解结果。

In [11]:
print(f"\n{'='*70}")
print("汇总结果")
print(f"{'='*70}\n")

print(f"{'初始货币':<10} {'最优解':<18} {'套利收益':<15} {'收益率':<12}")
print("-" * 70)

for s in currencies:
    name = currency_names[s]
    if results[s]['status'] == 'OPTIMAL':
        obj_val = results[s]['obj_val']
        profit = results[s]['profit']
        profit_rate = results[s]['profit_rate']
        print(f"{name:<10} {obj_val:<18.10f} {profit:<15.10f} {profit_rate:>10.6f}%")
    else:
        print(f"{name:<10} {results[s]['status']:<18} {'--':<15} {'--':>10}")

print("\n")


汇总结果

初始货币       最优解                套利收益            收益率         
----------------------------------------------------------------------
美元         1.0007001225       0.0007001225      0.070012%
欧元         1.0003211499       0.0003211499      0.032115%
英镑         1.0003083554       0.0003083554      0.030836%
日元         1.0007001225       0.0007001225      0.070012%




## 6. 详细分析某个特定货币的兑换路径（可选）

选择一个初始货币，详细查看其在每个时期的货币持有量变化。

In [12]:
# 选择要分析的初始货币（例如：1=美元）
analyze_currency = 1

if results[analyze_currency]['status'] == 'OPTIMAL':
    print(f"\n详细分析: 初始货币为{currency_names[analyze_currency]}")
    print("="*70)
    
    y = results[analyze_currency]['y']
    
    print("\n各时期货币持有量:")
    for t in range(T + 1):
        print(f"\n时期 t={t}:")
        for i in currencies:
            y_val = y[i, t].X
            if y_val > 1e-6:
                print(f"  {currency_names[i]}: {y_val:.6f}")


详细分析: 初始货币为美元

各时期货币持有量:

时期 t=0:
  美元: 1.000000

时期 t=1:
  日元: 133.380000

时期 t=2:
  美元: 1.000350

时期 t=3:
  日元: 133.426683

时期 t=4:
  美元: 1.000700


## 总结

本notebook展示了如何使用COPT求解器来解决多时期外汇套利问题的完整流程：

1. **问题建模**: 定义决策变量、目标函数和约束条件
2. **创建模型**: 使用COPT API构建优化模型
3. **求解模型**: 调用求解器获得最优解
4. **结果分析**: 提取和展示求解结果

### 关键要点

- **决策变量**: 使用`model.addVar()`添加变量
- **约束条件**: 使用`model.addConstr()`添加约束
- **目标函数**: 使用`model.setObjective()`设置优化目标
- **求解**: 使用`model.solve()`求解模型
- **结果提取**: 通过`model.objVal`和变量的`.X`属性获取结果

### 扩展练习

1. 尝试修改时期数T，观察套利收益的变化
2. 修改汇率矩阵，分析不同汇率对套利机会的影响
3. 添加交易成本约束，使模型更贴近实际情况